# Lecture 26: Scratch Implementations — Master Solutions & Execution Traces

Complete, verified implementations with execution traces and benchmarks.

## Problem 01: Vectorized Batch Gradient Descent Linear Regression

In [1]:
import numpy as np

class LinearRegressionGD:
    """Production-grade Linear Regression with Vectorized Batch Gradient Descent."""
    def __init__(self, lr: float = 0.01, n_iter: int = 1000):
        self.lr = lr
        self.n_iter = n_iter
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LinearRegressionGD":
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []

        for i in range(self.n_iter):
            y_pred = self.bias + np.dot(X, self.weights)
            residuals = y_pred - y

            cost = (1.0 / (2.0 * m)) * np.sum(residuals ** 2)
            self.loss_history.append(cost)

            db = (1.0 / m) * np.sum(residuals)
            dw = (1.0 / m) * np.dot(X.T, residuals)

            self.bias -= self.lr * db
            self.weights -= self.lr * dw

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.bias + np.dot(X, self.weights)

# Test on synthetic dataset
np.random.seed(42)
X_test = 2 * np.random.rand(150, 2)
y_test = 3.5 * X_test[:, 0] + 1.8 * X_test[:, 1] + 4.2 + np.random.randn(150) * 0.05

model = LinearRegressionGD(lr=0.05, n_iter=1500).fit(X_test, y_test)
print(f"Recovered Weights: {model.weights} (Expected: [3.5, 1.8])")
print(f"Recovered Bias:    {model.bias:.4f} (Expected: 4.2)")
print(f"Initial Cost:      {model.loss_history[0]:.4f}")
print(f"Final Cost:        {model.loss_history[-1]:.6f}")

Recovered Weights: [3.49868587 1.79443618] (Expected: [3.5, 1.8])
Recovered Bias:    4.2098 (Expected: 4.2)
Initial Cost:      47.6728
Final Cost:        0.001067


## Problem 02: Analytical Normal Equation with L2 Ridge Regularization

In [2]:
class LinearRegressionRidgeOLS:
    """Analytical Closed-Form Linear Regression with L2 Ridge Regularization."""
    def __init__(self, alpha: float = 1e-4):
        self.alpha = alpha
        self.bias = None
        self.weights = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LinearRegressionRidgeOLS":
        m, n = X.shape
        X_b = np.c_[np.ones((m, 1)), X]

        I_reg = np.eye(n + 1)
        I_reg[0, 0] = 0.0

        A = X_b.T @ X_b + self.alpha * I_reg
        b_vec = X_b.T @ y
        theta = np.linalg.solve(A, b_vec)

        self.bias = theta[0]
        self.weights = theta[1:]
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.bias + np.dot(X, self.weights)

ols_ridge = LinearRegressionRidgeOLS(alpha=0.01).fit(X_test, y_test)
print(f"Ridge OLS Weights: {ols_ridge.weights}")
print(f"Ridge OLS Bias:    {ols_ridge.bias:.4f}")
r2_ols = 1 - np.var(y_test - ols_ridge.predict(X_test)) / np.var(y_test)
print(f"Ridge OLS R2:      {r2_ols:.6f}")

Ridge OLS Weights: [3.49789365 1.79390782]
Ridge OLS Bias:    4.2111
Ridge OLS R2:      0.999616


## Problem 03: Vectorized Logistic Regression with Numerical Safeguards

In [3]:
class LogisticRegressionL2:
    """Vectorized Logistic Regression with Numerical Clipping and L2 Regularization."""
    def __init__(self, lr: float = 0.05, n_iter: int = 2000, l2_reg: float = 0.01):
        self.lr = lr
        self.n_iter = n_iter
        self.l2_reg = l2_reg
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionL2":
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        eps = 1e-15

        for _ in range(self.n_iter):
            z = self.bias + np.dot(X, self.weights)
            y_pred = self._sigmoid(z)
            
            y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
            loss = - (1.0 / m) * np.sum(y * np.log(y_pred_clipped) + (1.0 - y) * np.log(1.0 - y_pred_clipped))
            loss += (self.l2_reg / (2.0 * m)) * np.sum(self.weights ** 2)
            self.loss_history.append(loss)

            residuals = y_pred - y
            db = (1.0 / m) * np.sum(residuals)
            dw = (1.0 / m) * np.dot(X.T, residuals) + (self.l2_reg / m) * self.weights

            self.bias -= self.lr * db
            self.weights -= self.lr * dw

        return self

    def get_probabilities(self, X: np.ndarray) -> np.ndarray:
        return self._sigmoid(self.bias + np.dot(X, self.weights))

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.get_probabilities(X) >= threshold).astype(int)

y_clf = (X_test[:, 0] + X_test[:, 1] > 2.0).astype(int)
logreg = LogisticRegressionL2(lr=0.1, n_iter=2500).fit(X_test, y_clf)
print(f"Logistic Regression Weights: {logreg.weights}")
print(f"Logistic Regression Bias:    {logreg.bias:.4f}")
print(f"Binary Classification Accuracy: {np.mean(logreg.predict(X_test) == y_clf) * 100:.2f}%")

Logistic Regression Weights: [4.04297689 3.48037005]
Logistic Regression Bias:    -7.5673
Binary Classification Accuracy: 94.67%


## Problem 04: Multi-Class Softmax Regression from Scratch

In [4]:
class SoftmaxRegression:
    """Multi-Class Generalized Logistic Model with Numerically Stable Softmax."""
    def __init__(self, lr: float = 0.1, n_iter: int = 1500):
        self.lr = lr
        self.n_iter = n_iter
        self.W = None
        self.b = None

    def _softmax(self, Z: np.ndarray) -> np.ndarray:
        shifted_Z = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(shifted_Z)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def fit(self, X: np.ndarray, y: np.ndarray) -> "SoftmaxRegression":
        m, d = X.shape
        K = len(np.unique(y))
        self.W = np.zeros((d, K))
        self.b = np.zeros(K)

        Y_one_hot = np.zeros((m, K))
        Y_one_hot[np.arange(m), y] = 1.0

        for _ in range(self.n_iter):
            Z = X @ self.W + self.b
            P = self._softmax(Z)
            residuals = P - Y_one_hot

            dW = (1.0 / m) * (X.T @ residuals)
            db = (1.0 / m) * np.sum(residuals, axis=0)

            self.W -= self.lr * dW
            self.b -= self.lr * db

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._softmax(X @ self.W + self.b)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.argmax(self.predict_proba(X), axis=1)

y_multi = (X_test[:, 0] * 2 + X_test[:, 1]).astype(int) % 3
smax = SoftmaxRegression(lr=0.1, n_iter=1500).fit(X_test, y_multi)
print(f"Softmax Multi-Class Accuracy: {np.mean(smax.predict(X_test) == y_multi) * 100:.2f}%")

Softmax Multi-Class Accuracy: 46.67%


## Problem 05: Generalized K-Nearest Neighbors Classifier

In [5]:
class KNNClassifierAdvanced:
    """Generalized Metric-Space KNN supporting Lp Minkowski Distance and Weighting."""
    def __init__(self, k: int = 3, p: float = 2.0, weights: str = "uniform"):
        self.k = k
        self.p = p
        self.weights = weights
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNNClassifierAdvanced":
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=int)
        return self

    def _compute_distances(self, x: np.ndarray) -> np.ndarray:
        diff = np.abs(self.X_train - x)
        if self.p == 1.0:
            return np.sum(diff, axis=1)
        elif self.p == 2.0:
            return np.sqrt(np.sum(diff ** 2, axis=1))
        else:
            return np.sum(diff ** self.p, axis=1) ** (1.0 / self.p)

    def _predict_single(self, x: np.ndarray) -> int:
        dists = self._compute_distances(x)
        nn_idx = np.argsort(dists)[:self.k]
        nn_labels = self.y_train[nn_idx]

        if self.weights == "uniform":
            counts = np.bincount(nn_labels)
            return np.argmax(counts)
        elif self.weights == "distance":
            nn_dists = dists[nn_idx]
            weights = 1.0 / (nn_dists + 1e-6)
            classes = np.unique(self.y_train)
            class_scores = {c: 0.0 for c in classes}
            for lbl, w in zip(nn_labels, weights):
                class_scores[lbl] += w
            return max(class_scores, key=class_scores.get)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._predict_single(x) for x in np.asarray(X)])

knn_l2 = KNNClassifierAdvanced(k=5, p=2.0).fit(X_test, y_clf)
knn_l1 = KNNClassifierAdvanced(k=5, p=1.0).fit(X_test, y_clf)
print(f"KNN (Euclidean L2) Accuracy: {np.mean(knn_l2.predict(X_test) == y_clf) * 100:.2f}%")
print(f"KNN (Manhattan L1) Accuracy: {np.mean(knn_l1.predict(X_test) == y_clf) * 100:.2f}%")

KNN (Euclidean L2) Accuracy: 95.33%
KNN (Manhattan L1) Accuracy: 95.33%


## Problem 06: K-Nearest Neighbors Continuous Regressor

In [6]:
class KNNRegressor:
    """Non-parametric Continuous Target Estimator using Metric Neighborhoods."""
    def __init__(self, k: int = 5, weights: str = "distance"):
        self.k = k
        self.weights = weights
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNNRegressor":
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=float)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        preds = []
        for x in np.asarray(X, dtype=float):
            dists = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            nn_idx = np.argsort(dists)[:self.k]
            nn_y = self.y_train[nn_idx]
            
            if self.weights == "uniform":
                preds.append(np.mean(nn_y))
            elif self.weights == "distance":
                nn_d = dists[nn_idx]
                w = 1.0 / (nn_d + 1e-6)
                preds.append(np.sum(w * nn_y) / np.sum(w))
        return np.array(preds)

knn_reg = KNNRegressor(k=5, weights="distance").fit(X_test, y_test)
preds_knn_reg = knn_reg.predict(X_test)
r2_knn = 1 - np.var(y_test - preds_knn_reg) / np.var(y_test)
print(f"KNN Regressor R2 Score: {r2_knn:.4f}")

KNN Regressor R2 Score: 1.0000


## Problem 07: Mini-Batch & Stochastic Gradient Descent Optimizer Engine

In [7]:
class MiniBatchGDOptimizer:
    """Flexible Batching Optimizer supporting B=m (Batch), B=1 (SGD), and 1<B<m (Mini-Batch)."""
    def __init__(self, lr: float = 0.01, epochs: int = 100, batch_size: int = 32):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "MiniBatchGDOptimizer":
        m, d = X.shape
        self.weights = np.zeros(d)
        self.bias = 0.0
        self.loss_history = []
        B = min(self.batch_size, m)

        for epoch in range(self.epochs):
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for start in range(0, m, B):
                end = min(start + B, m)
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]
                batch_m = end - start

                y_pred = X_batch @ self.weights + self.bias
                res = y_pred - y_batch

                dw = (1.0 / batch_m) * (X_batch.T @ res)
                db = (1.0 / batch_m) * np.sum(res)

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

            full_pred = X @ self.weights + self.bias
            epoch_loss = (1.0 / (2.0 * m)) * np.sum((full_pred - y) ** 2)
            self.loss_history.append(epoch_loss)

        return self

opt_batch = MiniBatchGDOptimizer(lr=0.01, epochs=100, batch_size=len(X_test)).fit(X_test, y_test)
opt_sgd = MiniBatchGDOptimizer(lr=0.01, epochs=100, batch_size=1).fit(X_test, y_test)
opt_mini = MiniBatchGDOptimizer(lr=0.01, epochs=100, batch_size=16).fit(X_test, y_test)

print(f"Batch GD Final Loss:      {opt_batch.loss_history[-1]:.6f}")
print(f"SGD Final Loss:           {opt_sgd.loss_history[-1]:.6f}")
print(f"Mini-Batch GD Final Loss: {opt_mini.loss_history[-1]:.6f}")

Batch GD Final Loss:      0.278182
SGD Final Loss:           0.001068
Mini-Batch GD Final Loss: 0.016615


## Problem 08: First-Principles Pipeline vs Scikit-Learn Parity Test Harness

In [8]:
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN

def run_parity_tests():
    np.random.seed(42)
    m, d = 250, 4

    # 1. Linear Regression Parity Test
    X_reg = np.random.randn(m, d)
    true_w = np.array([2.5, -1.2, 0.8, 3.1])
    y_reg = X_reg @ true_w + 1.7 + np.random.randn(m) * 0.05

    our_ols = LinearRegressionRidgeOLS(alpha=0.0).fit(X_reg, y_reg)
    sk_ols = SklearnLR().fit(X_reg, y_reg)

    diff_w = np.max(np.abs(our_ols.weights - sk_ols.coef_))
    diff_b = np.abs(our_ols.bias - sk_ols.intercept_)
    assert diff_w < 1e-4 and diff_b < 1e-4, f"Linear Regression parity failed: diff_w={diff_w}, diff_b={diff_b}"
    print(f"PASS: Linear Regression Parity Confirmed (Max Error = {diff_w:.8f})")

    # 2. Logistic Regression Parity Test
    y_clf_p = (X_reg[:, 0] + X_reg[:, 1] > 0).astype(int)
    our_logreg = LogisticRegressionL2(lr=0.1, n_iter=3000, l2_reg=0.0).fit(X_reg, y_clf_p)
    sk_logreg = SklearnLogReg(penalty=None, solver='lbfgs').fit(X_reg, y_clf_p)

    acc_our = np.mean(our_logreg.predict(X_reg) == y_clf_p)
    acc_sk = np.mean(sk_logreg.predict(X_reg) == y_clf_p)
    assert abs(acc_our - acc_sk) < 0.02, f"Logistic parity failed: our={acc_our}, sk={acc_sk}"
    print(f"PASS: Logistic Regression Parity Confirmed (Acc: {acc_our*100:.2f}% vs Sklearn: {acc_sk*100:.2f}%)")

    # 3. KNN Parity Test
    our_knn = KNNClassifierAdvanced(k=5, p=2.0).fit(X_reg, y_clf_p)
    sk_knn = SklearnKNN(n_neighbors=5, p=2).fit(X_reg, y_clf_p)
    match_rate = np.mean(our_knn.predict(X_reg) == sk_knn.predict(X_reg))
    assert match_rate == 1.0, f"KNN mismatch rate: {1.0 - match_rate}"
    print(f"PASS: KNN Classifier Exact Match Confirmed (100.0% Equality)")

run_parity_tests()

PASS: Linear Regression Parity Confirmed (Max Error = 0.00000000)


PASS: Logistic Regression Parity Confirmed (Acc: 100.00% vs Sklearn: 100.00%)
PASS: KNN Classifier Exact Match Confirmed (100.0% Equality)


C:\Users\Shivam Keshari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
